# Assignment 4 — Part 2 setup and Q4 statistics

The Part 2 task is **natural language to SQL with `google-t5/t5-small`**. This notebook only sets up the separate environment and measures Q4; it does not train a model, fill the homework TODOs, or modify files in `part1/` or `part2/`.

Select your existing Colab session if it is still alive. Right-click the local **part2** folder in VS Code and choose **Upload to Colab**; the expected location is `/content/part2`. You may use a CPU runtime for Q4, although later T5 training needs a GPU. Do not create a second paid GPU session unnecessarily.

Run the following four code cells individually in order. Cell 2 downloads/installs the pinned Part 2 packages and may take several minutes. All homework commands use `/content/hw4-part2-env/bin/python`, leaving the Part 1 environment separate. Re-run setup if Colab replaces the runtime.

## 1. Check the Part 2 upload

This verifies the required files and paired train/dev line counts. It never imports the incomplete homework modules.

In [1]:
from pathlib import Path
import subprocess
import sys

assert sys.platform == "linux" and Path("/content").exists(), (
    "Select your Colab runtime, not local Mac Python."
)
PART2 = Path("/content/part2")
ENV2 = Path("/content/hw4-part2-env")
PYTHON2 = str(ENV2 / "bin/python")
ARTIFACTS2 = Path("/content/hw4-part2-artifacts")
ARTIFACTS2.mkdir(exist_ok=True)

for name in [
    "requirements.txt", "load_data.py", "t5_utils.py", "train_t5.py",
    "utils.py", "evaluate.py", "data/train.nl", "data/train.sql",
    "data/dev.nl", "data/dev.sql", "data/test.nl",
    "data/flight_database.db", "records/ground_truth_dev.pkl",
]:
    assert (PART2 / name).is_file(), (
        f"Missing {name}. Upload your local part2 folder to /content/part2."
    )

def run_p2_command(command, cwd=None, log=None):
    print("\nRunning:", " ".join(map(str, command)), flush=True)
    process = subprocess.Popen(
        list(map(str, command)), cwd=cwd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    try:
        for line in process.stdout:
            if log is not None:
                log.write(line)
                log.flush()
            print(line, end="", flush=True)
        code = process.wait()
    except BaseException:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        raise
    finally:
        process.stdout.close()
    if code:
        raise RuntimeError(f"Command failed with exit code {code}. See output above.")

for split in ["train", "dev"]:
    x = (PART2 / "data" / f"{split}.nl").read_text().splitlines()
    y = (PART2 / "data" / f"{split}.sql").read_text().splitlines()
    assert x and len(x) == len(y)
    assert all(s.strip() for s in x + y)
    print(f"{split}: {len(x)} aligned NL/SQL examples")
print("Part 2 requirements:\n" + (PART2 / "requirements.txt").read_text())
print("UPLOAD CHECK PASSED. No training has started.")


train: 4225 aligned NL/SQL examples
dev: 466 aligned NL/SQL examples
Part 2 requirements:
numpy==1.26.0
torch==2.1.2
tokenizers==0.19.1
transformers==4.40.0
accelerate==0.29.3
nltk==3.8.1
tqdm==4.66.1
pytest==7.4.4
hypothesis==6.96.1
wandb==0.15.10
matplotlib==3.8.0
plotly==5.18.0
seaborn==0.13.2
bitsandbytes==0.43.1
sentencepiece==0.2.0

UPLOAD CHECK PASSED. No training has started.


## 2. Create the separate Python 3.11 environment

Install the exact packages from the uploaded `part2/requirements.txt`. Do not edit its pins or install these packages into `hw4-part1-env`. Progress is streamed live.

In [2]:
# These commands use a NEW environment. Do not install Part 2 into the Part 1 env.
run_p2_command([sys.executable, "-m", "pip", "install", "uv"])
UV2 = [sys.executable, "-m", "uv"]
run_p2_command(UV2 + ["python", "install", "3.11"])
if not Path(PYTHON2).is_file():
    run_p2_command(UV2 + ["venv", "--python", "3.11", "--seed", str(ENV2)])
run_p2_command(UV2 + [
    "pip", "install", "--python", PYTHON2, "-r", str(PART2 / "requirements.txt"),
])
run_p2_command(UV2 + ["pip", "check", "--python", PYTHON2])
print("\nPART 2 PACKAGES INSTALLED")



Running: /usr/bin/python3 -m pip install uv

Running: /usr/bin/python3 -m uv python install 3.11
Python 3.11 is already installed

Running: /usr/bin/python3 -m uv venv --python 3.11 --seed /content/hw4-part2-env
Using CPython 3.11.18
Creating virtual environment with seed packages at: hw4-part2-env
 + packaging==26.3
 + pip==26.3
 + setuptools==84.2.1
 + wheel==0.48.0
Activate with: source hw4-part2-env/bin/activate

Running: /usr/bin/python3 -m uv pip install --python /content/hw4-part2-env/bin/python -r /content/part2/requirements.txt
Using Python 3.11.18 environment at: hw4-part2-env
Resolved 76 packages in 972ms
Prepared 37 packages in 7.61s
Installed 74 packages in 866ms
 + accelerate==0.29.3
 + appdirs==1.4.4
 + attrs==26.1.0
 + bitsandbytes==0.43.1
 + certifi==2026.10.23
 + charset-normalizer==3.5.2
 + click==8.5.1
 + cloudpickle==3.1.2
 + contourpy==1.3.3
 + cycler==0.12.1
 + docker-pycreds==0.4.0
 + filelock==4.2.10
 + fonttools==4.68.0
 + fsspec==2026.9.0
 + gitdb==4.0.12
 +

## 3. Verify the pinned environment

Check all required versions, imports, and (if available) a real CUDA tensor operation. An absent GPU does not block Q4. The package freeze and environment manifest will be included in the Q4 backup.

In [3]:
ENV2_CHECK = "import sys\nimport json\nimport importlib.metadata\nfrom pathlib import Path\nfrom packaging.requirements import Requirement\nimport torch\nimport transformers\nimport tokenizers\nimport numpy\nimport sentencepiece\n\nassert sys.version_info[:2] == (3, 11)\nrequirements = Path(sys.argv[1]).read_text().splitlines()\nversions = {}\nfor line in requirements:\n    line = line.strip()\n    if not line or line.startswith(\"#\"):\n        continue\n    requirement = Requirement(line)\n    actual = importlib.metadata.version(requirement.name)\n    assert requirement.specifier.contains(actual, prereleases=True), (\n        f\"{requirement.name}: expected {requirement.specifier}, found {actual}\"\n    )\n    versions[requirement.name] = actual\n    print(f\"{requirement.name}: {actual}\")\n\ngpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None\nif gpu:\n    tensor = torch.ones(4, device=\"cuda\")\n    assert (tensor + tensor).sum().item() == 8\n    print(\"GPU:\", gpu)\nelse:\n    print(\"CPU runtime: Q4 statistics work, but later model training requires a GPU.\")\n\nmetadata = {\n    \"python\": sys.version, \"executable\": sys.executable,\n    \"gpu\": gpu, \"versions\": versions,\n}\nPath(sys.argv[2]).write_text(json.dumps(metadata, indent=2))\nprint(\"PART 2 ENVIRONMENT READY\")\n"
run_p2_command([
    PYTHON2, "-u", "-c", ENV2_CHECK,
    str(PART2 / "requirements.txt"), str(ARTIFACTS2 / "environment.json"),
])
freeze = subprocess.run(
    [PYTHON2, "-m", "pip", "freeze"], capture_output=True, text=True, check=True,
)
(ARTIFACTS2 / "pip_freeze.txt").write_text(freeze.stdout)



Running: /content/hw4-part2-env/bin/python -u -c import sys
import json
import importlib.metadata
from pathlib import Path
from packaging.requirements import Requirement
import torch
import transformers
import tokenizers
import numpy
import sentencepiece

assert sys.version_info[:2] == (3, 11)
requirements = Path(sys.argv[1]).read_text().splitlines()
versions = {}
for line in requirements:
    line = line.strip()
    if not line or line.startswith("#"):
        continue
    requirement = Requirement(line)
    actual = importlib.metadata.version(requirement.name)
    assert requirement.specifier.contains(actual, prereleases=True), (
        f"{requirement.name}: expected {requirement.specifier}, found {actual}"
    )
    versions[requirement.name] = actual
    print(f"{requirement.name}: {actual}")

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
if gpu:
    tensor = torch.ones(4, device="cuda")
    assert (tensor + tensor).sum().item() == 8
    print("GPU:",

1423

## 4. Measure Q4 and save to Drive

Finish Drive authorization before proceeding. This cell loads **only the T5 tokenizer**, not the T5 model. It measures train and dev only, using the full sequences without padding or truncation.

**Definitions:** mean length = emitted T5 subword tokens per example, excluding automatically added EOS/BOS and padding (`add_special_tokens=False`). Vocabulary size = distinct emitted token IDs observed separately in each split and modality, including the unknown token if it occurs; this is not the tokenizer's entire fixed vocabulary.

**Initial preprocessing choice:** preserve the data content; strip outer whitespace only, consistent with the starter `load_lines()` helper. No lowercasing, task prefix, SQL rewriting, deduplication, or filtering. The supplied data have no outer whitespace, so the two tables are expected to match. This is permitted by Q4. If the actual training preprocessing later changes, update the helper and regenerate Table 2.

The output also audits maximum lengths and counts over 512 tokens, to inform later length settings. It does not silently truncate long SQL queries. Saved files include `q4_statistics.json`, the two table CSVs, `q4_tables.tex`, the helper source, input hashes, environment versions, logs, and status. The LaTeX tables use the `Y` column already defined in your Overleaf report.

In [4]:
from google.colab import drive
from datetime import datetime, timezone
import json
import shutil
import traceback

drive.mount("/content/drive")
Q4_BACKUP = (
    Path("/content/drive/MyDrive/NYU_NLP_HW4/part2/q4")
    / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
)
Q4_BACKUP.mkdir(parents=True, exist_ok=False)
print("Q4 BACKUP FOLDER:", Q4_BACKUP, flush=True)

for name in ["environment.json", "pip_freeze.txt"]:
    shutil.copy2(ARTIFACTS2 / name, Q4_BACKUP / name)
(Q4_BACKUP / "source").mkdir()
for name in [
    "requirements.txt", "load_data.py", "t5_utils.py", "train_t5.py",
    "utils.py", "evaluate.py",
]:
    shutil.copy2(PART2 / name, Q4_BACKUP / "source" / name)

Q4_SOURCE = "\"\"\"Q4 measurement helper. Reads data only; does not import homework skeletons.\"\"\"\n\nimport argparse\nimport csv\nimport hashlib\nimport importlib.metadata\nimport json\nfrom pathlib import Path\nfrom statistics import mean\n\n\nMODEL_NAME = \"google-t5/t5-small\"\nROWS = [\n    (\"Number of examples\", \"examples\"),\n    (\"Mean sentence length (T5 tokens)\", \"mean_sentence_length\"),\n    (\"Mean SQL query length (T5 tokens)\", \"mean_sql_length\"),\n    (\"Vocabulary size (natural language)\", \"nl_vocabulary_size\"),\n    (\"Vocabulary size (SQL)\", \"sql_vocabulary_size\"),\n]\n\n\ndef read_split(data_dir, split):\n    # Keep every example, including duplicate examples, in its original order.\n    inputs = (data_dir / f\"{split}.nl\").read_text(encoding=\"utf-8\").splitlines()\n    targets = (data_dir / f\"{split}.sql\").read_text(encoding=\"utf-8\").splitlines()\n    assert inputs and len(inputs) == len(targets), f\"Unaligned {split} files\"\n    assert all(x.strip() for x in inputs + targets), f\"Blank example in {split}\"\n    return inputs, targets\n\n\ndef preprocess(text):\n    # Baseline: match load_lines()'s outer-whitespace stripping only.\n    # Preserve case, internal whitespace, SQL literals, punctuation, and order.\n    # If the training preprocessing changes later, update this and rerun Q4.\n    return text.strip()\n\n\ndef text_statistics(tokenizer, texts, batch_size=256):\n    lengths = []\n    vocabulary = set()\n    unknown_count = 0\n    for start in range(0, len(texts), batch_size):\n        encoded = tokenizer(\n            texts[start:start + batch_size],\n            add_special_tokens=False,\n            padding=False,\n            truncation=False,\n            return_attention_mask=False,\n            return_token_type_ids=False,\n            verbose=False,\n        )[\"input_ids\"]\n        assert len(encoded) == len(texts[start:start + batch_size])\n        for ids in encoded:\n            lengths.append(len(ids))\n            vocabulary.update(ids)\n            unknown_count += ids.count(tokenizer.unk_token_id)\n    assert len(lengths) == len(texts)\n    return {\n        \"mean_length\": mean(lengths),\n        \"vocabulary_size\": len(vocabulary),\n        \"minimum_length\": min(lengths),\n        \"maximum_length\": max(lengths),\n        \"examples_over_512_tokens\": sum(n > 512 for n in lengths),\n        \"unknown_tokens\": unknown_count,\n        \"total_tokens\": sum(lengths),\n    }\n\n\ndef paired_statistics(tokenizer, inputs, targets):\n    nl = text_statistics(tokenizer, inputs)\n    sql = text_statistics(tokenizer, targets)\n    return {\n        \"examples\": len(inputs),\n        \"mean_sentence_length\": nl[\"mean_length\"],\n        \"mean_sql_length\": sql[\"mean_length\"],\n        \"nl_vocabulary_size\": nl[\"vocabulary_size\"],\n        \"sql_vocabulary_size\": sql[\"vocabulary_size\"],\n        \"input_audit\": nl,\n        \"target_audit\": sql,\n    }\n\n\ndef collect_statistics(tokenizer, data_dir):\n    before, after, changes = {}, {}, {}\n    for split in [\"train\", \"dev\"]:\n        inputs, targets = read_split(data_dir, split)\n        print(f\"Measuring {split}: {len(inputs)} aligned pairs\", flush=True)\n        processed_inputs = [preprocess(s) for s in inputs]\n        processed_targets = [preprocess(s) for s in targets]\n        before[split] = paired_statistics(tokenizer, inputs, targets)\n        after[split] = paired_statistics(tokenizer, processed_inputs, processed_targets)\n        changes[split] = {\n            \"changed_nl_lines\": sum(a != b for a, b in zip(inputs, processed_inputs)),\n            \"changed_sql_lines\": sum(a != b for a, b in zip(targets, processed_targets)),\n        }\n    return before, after, changes\n\n\ndef formatted(value):\n    return f\"{value:.2f}\" if isinstance(value, float) else str(value)\n\n\ndef table_rows(splits, include_examples):\n    return [\n        [label, formatted(splits[\"train\"][key]), formatted(splits[\"dev\"][key])]\n        for label, key in ROWS if include_examples or key != \"examples\"\n    ]\n\n\ndef render_latex_table(rows, caption, label):\n    lines = [\n        r\"\\begin{table}[htbp]\", r\"\\centering\",\n        r\"\\begin{tabularx}{\\linewidth}{Yrr}\", r\"\\toprule\",\n        r\"Statistic & Train & Dev \\\\\", r\"\\midrule\",\n    ]\n    lines.extend(\" & \".join(row) + r\" \\\\\" for row in rows)\n    lines.extend([\n        r\"\\bottomrule\", r\"\\end{tabularx}\",\n        \"\\\\caption{\" + caption + \"}\", \"\\\\label{\" + label + \"}\",\n        r\"\\end{table}\",\n    ])\n    return \"\\n\".join(lines)\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--data-dir\", type=Path, required=True)\n    parser.add_argument(\"--output-dir\", type=Path, required=True)\n    args = parser.parse_args()\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n\n    from transformers import T5TokenizerFast\n\n    print(\"Loading tokenizer only; no model weights or training.\", flush=True)\n    tokenizer = T5TokenizerFast.from_pretrained(MODEL_NAME)\n    before, after, changes = collect_statistics(tokenizer, args.data_dir)\n    expected = {\"train\": 4225, \"dev\": 466}\n    for split, size in expected.items():\n        assert before[split][\"examples\"] == size, f\"Unexpected {split} example count\"\n\n    report = {\n        \"tokenizer_checkpoint\": MODEL_NAME,\n        \"tokenizer_class\": type(tokenizer).__name__,\n        \"tokenizer_size_for_reference_not_observed_vocabulary\": len(tokenizer),\n        \"versions\": {name: importlib.metadata.version(name)\n                     for name in [\"transformers\", \"tokenizers\", \"sentencepiece\"]},\n        \"conventions\": {\n            \"mean_lengths\": \"Mean emitted subword-token count per example; before truncation.\",\n            \"special_tokens\": \"No added EOS/BOS or padding; add_special_tokens=False.\",\n            \"vocabulary\": \"Distinct emitted token IDs, separately per split and modality; includes UNK if emitted.\",\n            \"preprocessing\": \"Outer-whitespace stripping only, matching load_lines(); no lowercasing, prefix, SQL rewriting, or example removal.\",\n            \"splits\": \"Train and dev only; no test-set analysis.\",\n        },\n        \"data_sha256\": {\n            name: hashlib.sha256((args.data_dir / name).read_bytes()).hexdigest()\n            for name in [\"train.nl\", \"train.sql\", \"dev.nl\", \"dev.sql\"]\n        },\n        \"changed_lines\": changes,\n        \"before\": before,\n        \"after\": after,\n    }\n    (args.output_dir / \"q4_statistics.json\").write_text(json.dumps(report, indent=2))\n\n    tex = []\n    for phase, splits, include_examples in [\n        (\"before\", before, True), (\"after\", after, False),\n    ]:\n        rows = table_rows(splits, include_examples)\n        with (args.output_dir / f\"q4_{phase}.csv\").open(\"w\", newline=\"\") as stream:\n            writer = csv.writer(stream)\n            writer.writerow([\"Statistic\", \"Train\", \"Dev\"])\n            writer.writerows(rows)\n        print(f\"\\nQ4 TABLE: {phase.upper()} PREPROCESSING\")\n        print(\"Statistic | Train | Dev\")\n        for row in rows:\n            print(\" | \".join(row))\n        tex.append(render_latex_table(\n            rows, f\"Data statistics {phase} preprocessing.\",\n            f\"tab:{phase}-preprocessing\",\n        ))\n    (args.output_dir / \"q4_tables.tex\").write_text(\"\\n\\n\".join(tex) + \"\\n\")\n\n    print(\"\\nLength audit (untruncated T5 tokens):\")\n    for split in [\"train\", \"dev\"]:\n        for modality, key in [(\"NL\", \"input_audit\"), (\"SQL\", \"target_audit\")]:\n            audit = after[split][key]\n            print(f\"{split} {modality}: max={audit['maximum_length']}; \"\n                  f\"over 512={audit['examples_over_512_tokens']}; \"\n                  f\"UNK tokens={audit['unknown_tokens']}\")\n    print(\"Changed lines:\", json.dumps(changes))\n    print(\"\\nQ4 STATISTICS SAVED:\", args.output_dir, flush=True)\n\n\nif __name__ == \"__main__\":\n    main()\n"

Q4_SCRIPT = ARTIFACTS2 / "q4_data_statistics.py"
Q4_SCRIPT.write_text(Q4_SOURCE)
shutil.copy2(Q4_SCRIPT, Q4_BACKUP / Q4_SCRIPT.name)

def q4_status(state, error=None):
    info = {
        "state": state,
        "updated_utc": datetime.now(timezone.utc).isoformat(),
        "backup_folder": str(Q4_BACKUP),
    }
    if error:
        info["error"] = error
    (Q4_BACKUP / "status.json").write_text(json.dumps(info, indent=2))

q4_status("RUNNING")
try:
    with (Q4_BACKUP / "q4.log").open("w", buffering=1) as log:
        run_p2_command([
            PYTHON2, "-u", str(Q4_SCRIPT),
            "--data-dir", str(PART2 / "data"),
            "--output-dir", str(Q4_BACKUP),
        ], cwd=PART2, log=log)
except BaseException:
    q4_status("FAILED", traceback.format_exc())
    print("Available logs/artifacts saved at:", Q4_BACKUP)
    raise
else:
    q4_status("COMPLETED")
    print("\nQ4 MEASUREMENT COMPLETE:", Q4_BACKUP)
    print("Send the two printed tables and the length audit for review.")
    print("Do not start train_t5.py yet.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Q4 BACKUP FOLDER: /content/drive/MyDrive/NYU_NLP_HW4/part2/q4/20261110T020247929743Z

Running: /content/hw4-part2-env/bin/python -u /content/hw4-part2-artifacts/q4_data_statistics.py --data-dir /content/part2/data --output-dir /content/drive/MyDrive/NYU_NLP_HW4/part2/q4/20261110T020247929743Z
Loading tokenizer only; no model weights or training.
/content/hw4-part2-env/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Measuring train: 4225 aligned pairs
Measuring dev: 466 aligned pairs

Q4 TABLE: BEFORE PREPROCESSING
Statistic | Train | Dev
Number of examples | 4225 | 466
Mean sentence length (T5 tokens) | 17.10 | 17.07
Mean SQL query length 

## 5. Optional: inspect the saved Q4 artifacts

This only reads the latest backup. After reviewing the actual measurements, place the Q4 description and tables between `BEGIN Q4_ANSWER` and `END Q4_ANSWER` in Overleaf. The gray draft instructions should be removed when the section is completed.

**Later Part 2 requirements:** Q5 describes the actual model processing, tokenization, architecture, and hyperparameters. Q6 reports dev experiments and final test results, plus at least three observed error classes with examples and COUNT/TOTAL. Q7 submits the final SQL and record files and requires a checkpoint-folder link in the written report. Select models on dev; preserve test order. The supplied training script still has placeholders and training defaults that must be configured before it is useful.

In [ ]:
# Read-only inspection; can also be run after reconnecting to a fresh runtime.
from google.colab import drive
from pathlib import Path
import json

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive/NYU_NLP_HW4/part2/q4")
runs = sorted(p for p in root.iterdir() if p.is_dir()) if root.exists() else []
assert runs, "No Q4 backup exists yet."
saved = runs[-1]
print("Inspecting:", saved)
print((saved / "status.json").read_text())
report_path = saved / "q4_statistics.json"
if report_path.exists():
    report = json.loads(report_path.read_text())
    for phase in ["before", "after"]:
        print(f"\n{phase.upper()}:")
        for split in ["train", "dev"]:
            print(split, json.dumps(report[phase][split], indent=2))
    print("\nChanged lines:", report["changed_lines"])
else:
    print("No statistics JSON yet. Inspect q4.log for progress/errors.")
print("\nSaved files:")
for path in sorted(saved.iterdir()):
    print(path.name)


In [5]:
tokenizer_check = """
from transformers import T5TokenizerFast

tokenizer = T5TokenizerFast.from_pretrained("google-t5/t5-small")

for text in ["<", "<=", ">", ">="]:
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    print(
        repr(text),
        tokenizer.convert_ids_to_tokens(ids),
        "contains UNK:", tokenizer.unk_token_id in ids,
    )
"""

run_p2_command([PYTHON2, "-u", "-c", tokenizer_check])


Running: /content/hw4-part2-env/bin/python -u -c 
from transformers import T5TokenizerFast

tokenizer = T5TokenizerFast.from_pretrained("google-t5/t5-small")

for text in ["<", "<=", ">", ">="]:
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    print(
        repr(text),
        tokenizer.convert_ids_to_tokens(ids),
        "contains UNK:", tokenizer.unk_token_id in ids,
    )

/content/hw4-part2-env/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
'<' ['▁', '<unk>'] contains UNK: True
'<=' ['▁', '<unk>', '='] contains UNK: True
'>' ['▁>'] contains UNK: False
'>=' ['▁>', '='] contains UNK: False


## 6. Finalize Q4: preserve the less-than operator and regenerate the tables

Run **only this new code cell** after the earlier setup has passed. It tests a proposed lossless representation using the unchanged T5 tokenizer: SQL `<` is encoded as the ordinary text marker `less_than`; `<=` consequently becomes `less_than=`. Restore that marker to `<` after decoding, before executing or evaluating generated SQL. This marker is not a special token and requires no new vocabulary or embedding resize.

The cell checks operator encode/tokenize/decode/restore examples, rejects marker collisions in the original train/dev SQL, checks exact inversion of the text preprocessing for every train/dev target, and requires zero unknown tokens in the processed SQL. It then recomputes the original and processed statistics, including the three requested extra rows, and saves a **new timestamped Drive backup**. No homework file or dataset is changed and no model is trained.

Use the resulting **new after-preprocessing table** instead of the earlier whitespace-only baseline table. Check the new maximum SQL lengths before choosing later training limits. The marker functions are saved with the measurement source; they must be used consistently when you implement the homework data loader and prediction postprocessing. This cell does not implement those training functions.

In [6]:
# Run this finalization cell only; it does not train a model.
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive
import json
import shutil
import traceback

PART2 = Path("/content/part2")
PYTHON2 = "/content/hw4-part2-env/bin/python"
ARTIFACTS2 = Path("/content/hw4-part2-artifacts")
assert Path(PYTHON2).is_file(), "Reconnect to the existing Part 2 runtime, or rerun setup."
assert callable(globals().get("run_p2_command")), "Run Cell 1 to define run_p2_command."
assert (ARTIFACTS2 / "environment.json").is_file(), "Run the environment verification cell."

drive.mount("/content/drive")
Q4_FINAL_BACKUP = (
    Path("/content/drive/MyDrive/NYU_NLP_HW4/part2/q4")
    / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
)
Q4_FINAL_BACKUP.mkdir(parents=True, exist_ok=False)
print("NEW Q4 BACKUP:", Q4_FINAL_BACKUP, flush=True)

for name in ["environment.json", "pip_freeze.txt"]:
    shutil.copy2(ARTIFACTS2 / name, Q4_FINAL_BACKUP / name)
(Q4_FINAL_BACKUP / "source").mkdir()
for name in [
    "requirements.txt", "load_data.py", "t5_utils.py", "train_t5.py",
    "utils.py", "evaluate.py",
]:
    shutil.copy2(PART2 / name, Q4_FINAL_BACKUP / "source" / name)

Q4_MARKER_SOURCE = "\"\"\"Q4 measurement helper. Reads data only; does not import homework skeletons.\"\"\"\n\nimport argparse\nimport csv\nimport hashlib\nimport importlib.metadata\nimport json\nfrom pathlib import Path\nfrom statistics import mean\n\n\nMODEL_NAME = \"google-t5/t5-small\"\nROWS = [\n    (\"Number of examples\", \"examples\"),\n    (\"Mean sentence length (T5 tokens)\", \"mean_sentence_length\"),\n    (\"Mean SQL query length (T5 tokens)\", \"mean_sql_length\"),\n    (\"Vocabulary size (natural language)\", \"nl_vocabulary_size\"),\n    (\"Vocabulary size (SQL)\", \"sql_vocabulary_size\"),\n    (\"Maximum sentence length (T5 tokens)\", \"max_sentence_length\"),\n    (\"Maximum SQL query length (T5 tokens)\", \"max_sql_length\"),\n    (\"SQL unknown-token occurrences\", \"sql_unknown_tokens\"),\n]\n\n\ndef read_split(data_dir, split):\n    # Keep every example, including duplicate examples, in its original order.\n    inputs = (data_dir / f\"{split}.nl\").read_text(encoding=\"utf-8\").splitlines()\n    targets = (data_dir / f\"{split}.sql\").read_text(encoding=\"utf-8\").splitlines()\n    assert inputs and len(inputs) == len(targets), f\"Unaligned {split} files\"\n    assert all(x.strip() for x in inputs + targets), f\"Blank example in {split}\"\n    return inputs, targets\n\n\ndef preprocess(text):\n    # Baseline: match load_lines()'s outer-whitespace stripping only.\n    # Preserve case, internal whitespace, SQL literals, punctuation, and order.\n    # If the training preprocessing changes later, update this and rerun Q4.\n    return text.strip()\n\n\ndef preprocess_sql(text, marker=None):\n    text = text.strip()\n    if marker is None:\n        return text\n    assert marker not in text, \"SQL marker collision: choose a different reserved marker.\"\n    return text.replace(\"<\", marker)\n\n\ndef restore_sql(text, marker):\n    return text.replace(marker, \"<\")\n\n\ndef validate_marker(tokenizer, marker):\n    probes = [\"<\", \"<=\", \">\", \">=\", \"x < 1200\", \"x <= 1200\"]\n    results = []\n    for text in probes:\n        encoded_text = preprocess_sql(text, marker)\n        ids = tokenizer(encoded_text, add_special_tokens=False)[\"input_ids\"]\n        assert tokenizer.unk_token_id not in ids, f\"Marker still yields UNK: {text}\"\n        decoded = tokenizer.decode(\n            ids, skip_special_tokens=True, clean_up_tokenization_spaces=False,\n        )\n        restored = restore_sql(decoded, marker).strip()\n        assert restored == text, f\"Tokenization round-trip failed: {text!r} -> {restored!r}\"\n        print(f\"Operator check: {text!r} -> {encoded_text!r} -> {restored!r}\", flush=True)\n        results.append({\"original\": text, \"encoded\": encoded_text, \"restored\": restored})\n    return results\n\n\ndef text_statistics(tokenizer, texts, batch_size=256):\n    lengths = []\n    vocabulary = set()\n    unknown_count = 0\n    for start in range(0, len(texts), batch_size):\n        encoded = tokenizer(\n            texts[start:start + batch_size],\n            add_special_tokens=False,\n            padding=False,\n            truncation=False,\n            return_attention_mask=False,\n            return_token_type_ids=False,\n            verbose=False,\n        )[\"input_ids\"]\n        assert len(encoded) == len(texts[start:start + batch_size])\n        for ids in encoded:\n            lengths.append(len(ids))\n            vocabulary.update(ids)\n            unknown_count += ids.count(tokenizer.unk_token_id)\n    assert len(lengths) == len(texts)\n    return {\n        \"mean_length\": mean(lengths),\n        \"vocabulary_size\": len(vocabulary),\n        \"minimum_length\": min(lengths),\n        \"maximum_length\": max(lengths),\n        \"examples_over_512_tokens\": sum(n > 512 for n in lengths),\n        \"unknown_tokens\": unknown_count,\n        \"total_tokens\": sum(lengths),\n    }\n\n\ndef paired_statistics(tokenizer, inputs, targets):\n    nl = text_statistics(tokenizer, inputs)\n    sql = text_statistics(tokenizer, targets)\n    return {\n        \"examples\": len(inputs),\n        \"mean_sentence_length\": nl[\"mean_length\"],\n        \"mean_sql_length\": sql[\"mean_length\"],\n        \"nl_vocabulary_size\": nl[\"vocabulary_size\"],\n        \"sql_vocabulary_size\": sql[\"vocabulary_size\"],\n        \"max_sentence_length\": nl[\"maximum_length\"],\n        \"max_sql_length\": sql[\"maximum_length\"],\n        \"sql_unknown_tokens\": sql[\"unknown_tokens\"],\n        \"input_audit\": nl,\n        \"target_audit\": sql,\n    }\n\n\ndef collect_statistics(tokenizer, data_dir, sql_marker=None):\n    before, after, changes = {}, {}, {}\n    for split in [\"train\", \"dev\"]:\n        inputs, targets = read_split(data_dir, split)\n        print(f\"Measuring {split}: {len(inputs)} aligned pairs\", flush=True)\n        processed_inputs = [preprocess(s) for s in inputs]\n        processed_targets = [preprocess_sql(s, sql_marker) for s in targets]\n        if sql_marker:\n            assert all(restore_sql(encoded, sql_marker) == raw.strip()\n                       for raw, encoded in zip(targets, processed_targets))\n        before[split] = paired_statistics(tokenizer, inputs, targets)\n        after[split] = paired_statistics(tokenizer, processed_inputs, processed_targets)\n        if sql_marker:\n            assert after[split][\"sql_unknown_tokens\"] == 0, (\n                f\"Unknown SQL tokens remain in {split}; do not finalize preprocessing.\"\n            )\n            print(f\"{split}: all {len(targets)} SQL text conversions are reversible; \"\n                  \"processed SQL has zero unknown tokens.\", flush=True)\n        changes[split] = {\n            \"changed_nl_lines\": sum(a != b for a, b in zip(inputs, processed_inputs)),\n            \"changed_sql_lines\": sum(a != b for a, b in zip(targets, processed_targets)),\n        }\n    return before, after, changes\n\n\ndef formatted(value):\n    return f\"{value:.2f}\" if isinstance(value, float) else str(value)\n\n\ndef table_rows(splits, include_examples):\n    return [\n        [label, formatted(splits[\"train\"][key]), formatted(splits[\"dev\"][key])]\n        for label, key in ROWS if include_examples or key != \"examples\"\n    ]\n\n\ndef render_latex_table(rows, caption, label):\n    lines = [\n        r\"\\begin{table}[htbp]\", r\"\\centering\",\n        r\"\\begin{tabularx}{\\linewidth}{Yrr}\", r\"\\toprule\",\n        r\"Statistic & Train & Dev \\\\\", r\"\\midrule\",\n    ]\n    lines.extend(\" & \".join(row) + r\" \\\\\" for row in rows)\n    lines.extend([\n        r\"\\bottomrule\", r\"\\end{tabularx}\",\n        \"\\\\caption{\" + caption + \"}\", \"\\\\label{\" + label + \"}\",\n        r\"\\end{table}\",\n    ])\n    return \"\\n\".join(lines)\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--data-dir\", type=Path, required=True)\n    parser.add_argument(\"--output-dir\", type=Path, required=True)\n    parser.add_argument(\"--sql-marker\", default=None,\n                        help=\"Encode '<' using this collision-checked ordinary text marker.\")\n    args = parser.parse_args()\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n\n    from transformers import T5TokenizerFast\n\n    print(\"Loading tokenizer only; no model weights or training.\", flush=True)\n    tokenizer = T5TokenizerFast.from_pretrained(MODEL_NAME)\n    probes = validate_marker(tokenizer, args.sql_marker) if args.sql_marker else []\n    before, after, changes = collect_statistics(tokenizer, args.data_dir, args.sql_marker)\n    expected = {\"train\": 4225, \"dev\": 466}\n    for split, size in expected.items():\n        assert before[split][\"examples\"] == size, f\"Unexpected {split} example count\"\n\n    report = {\n        \"tokenizer_checkpoint\": MODEL_NAME,\n        \"tokenizer_class\": type(tokenizer).__name__,\n        \"tokenizer_size_for_reference_not_observed_vocabulary\": len(tokenizer),\n        \"versions\": {name: importlib.metadata.version(name)\n                     for name in [\"transformers\", \"tokenizers\", \"sentencepiece\"]},\n        \"conventions\": {\n            \"mean_lengths\": \"Mean emitted subword-token count per example; before truncation.\",\n            \"special_tokens\": \"No added EOS/BOS or padding; add_special_tokens=False.\",\n            \"vocabulary\": \"Distinct emitted token IDs, separately per split and modality; includes UNK if emitted.\",\n            \"preprocessing\": (\n                \"Outer-whitespace stripping only, matching load_lines(); no lowercasing, task prefix, or example removal.\"\n                if not args.sql_marker else\n                f\"NL: outer-whitespace stripping only. SQL: strip outer whitespace, replace '<' by {args.sql_marker!r}; restore the marker to '<' after decoding and before SQL execution/evaluation. No tokenizer vocabulary changes, lowercasing, task prefix, or example removal.\"\n            ),\n            \"splits\": \"Train and dev only; no test-set analysis.\",\n        },\n        \"data_sha256\": {\n            name: hashlib.sha256((args.data_dir / name).read_bytes()).hexdigest()\n            for name in [\"train.nl\", \"train.sql\", \"dev.nl\", \"dev.sql\"]\n        },\n        \"changed_lines\": changes,\n        \"sql_marker\": args.sql_marker,\n        \"operator_round_trip_checks\": probes,\n        \"before\": before,\n        \"after\": after,\n    }\n    (args.output_dir / \"q4_statistics.json\").write_text(json.dumps(report, indent=2))\n\n    tex = []\n    for phase, splits, include_examples in [\n        (\"before\", before, True), (\"after\", after, False),\n    ]:\n        rows = table_rows(splits, include_examples)\n        with (args.output_dir / f\"q4_{phase}.csv\").open(\"w\", newline=\"\") as stream:\n            writer = csv.writer(stream)\n            writer.writerow([\"Statistic\", \"Train\", \"Dev\"])\n            writer.writerows(rows)\n        print(f\"\\nQ4 TABLE: {phase.upper()} PREPROCESSING\")\n        print(\"Statistic | Train | Dev\")\n        for row in rows:\n            print(\" | \".join(row))\n        tex.append(render_latex_table(\n            rows, f\"Data statistics {phase} preprocessing.\",\n            f\"tab:{phase}-preprocessing\",\n        ))\n    (args.output_dir / \"q4_tables.tex\").write_text(\"\\n\\n\".join(tex) + \"\\n\")\n\n    print(\"\\nLength audit (untruncated T5 tokens):\")\n    for split in [\"train\", \"dev\"]:\n        for modality, key in [(\"NL\", \"input_audit\"), (\"SQL\", \"target_audit\")]:\n            audit = after[split][key]\n            print(f\"{split} {modality}: max={audit['maximum_length']}; \"\n                  f\"over 512={audit['examples_over_512_tokens']}; \"\n                  f\"UNK tokens={audit['unknown_tokens']}\")\n    print(\"Changed lines:\", json.dumps(changes))\n    print(\"\\nQ4 STATISTICS SAVED:\", args.output_dir, flush=True)\n\n\nif __name__ == \"__main__\":\n    main()\n"

marker_script = ARTIFACTS2 / "q4_data_statistics_marker.py"
marker_script.write_text(Q4_MARKER_SOURCE)
shutil.copy2(marker_script, Q4_FINAL_BACKUP / marker_script.name)

def marker_status(state, error=None):
    status = {
        "state": state,
        "variant": "SQL '<' represented by ordinary text marker 'less_than'",
        "updated_utc": datetime.now(timezone.utc).isoformat(),
        "backup_folder": str(Q4_FINAL_BACKUP),
    }
    if error:
        status["error"] = error
    (Q4_FINAL_BACKUP / "status.json").write_text(json.dumps(status, indent=2))

marker_status("RUNNING")
try:
    with (Q4_FINAL_BACKUP / "q4.log").open("w", buffering=1) as log:
        run_p2_command([
            PYTHON2, "-u", str(marker_script),
            "--data-dir", str(PART2 / "data"),
            "--output-dir", str(Q4_FINAL_BACKUP),
            "--sql-marker", "less_than",
        ], cwd=PART2, log=log)
except BaseException:
    marker_status("FAILED", traceback.format_exc())
    print("Check failed. Saved log:", Q4_FINAL_BACKUP / "q4.log")
    raise
else:
    marker_status("COMPLETED")
    print("\nQ4 PREPROCESSING CHECKS PASSED")
    print("SAVE FOLDER:", Q4_FINAL_BACKUP)
    print("Save this notebook and send the new tables/length audit for review.")
    print("The same encode/restore rule must later be used in training and inference.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
NEW Q4 BACKUP: /content/drive/MyDrive/NYU_NLP_HW4/part2/q4/20261110T022316884761Z

Running: /content/hw4-part2-env/bin/python -u /content/hw4-part2-artifacts/q4_data_statistics_marker.py --data-dir /content/part2/data --output-dir /content/drive/MyDrive/NYU_NLP_HW4/part2/q4/20261110T022316884761Z --sql-marker less_than
Loading tokenizer only; no model weights or training.
/content/hw4-part2-env/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Operator check: '<' -> 'less_than' -> '<'
Operator check: '<=' -> 'less_than=' -> '<='
Operator check: '>' -> '>' -> '>'
Operator check: '>=' -> '>=' -> '>='
Operator check: 'x < 1200' -> 'x less_than 

## 7. Q5 first step: data loading and batching

First open the local `support/q5_data_loader_reference.py`. In `part2/load_data.py`, replace the block from `class T5Dataset(Dataset):` up to, **but not including**, `def get_dataloader(batch_size, split):` with the reference block. Keep the existing imports, `PAD_IDX = 0`, and all functions starting with `get_dataloader` below it. Save, then upload the updated `part2` folder to Colab.

The loader uses the Q4 `less_than` SQL representation, appends EOS without truncation, uses dynamic right padding, and uses the standard T5 PAD decoder-start token. Target padding remains 0 to match the supplied training loop's loss mask. Train/dev batches return five tensors; test batches return three tensors, without reading nonexistent test SQL labels. The reference does not change the tokenizer vocabulary.

Run only the following new code cell after uploading. It checks the real uploaded module, including all split counts, EOS, teacher-forcing shifts, padding/masks, unknown SQL tokens, Q4-to-loader length consistency, SQL operator round trips, and dev/test ordering. It saves the source and check logs to a new Drive folder. It does not initialize model weights or train. Expected loader sizes with batch size 8: train 529 batches, dev 59, test 54.

In [7]:
# Run AFTER pasting the reference block into load_data.py, saving, and uploading part2.
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive
import json
import shutil
import traceback

PART2 = Path("/content/part2")
PYTHON2 = "/content/hw4-part2-env/bin/python"
ARTIFACTS2 = Path("/content/hw4-part2-artifacts")
assert Path(PYTHON2).is_file(), "Use the existing Part 2 runtime, or rerun its setup."
assert callable(globals().get("run_p2_command")), "Run Cell 1 to define run_p2_command."
assert (PART2 / "load_data.py").is_file(), "Upload the updated part2 folder first."
ARTIFACTS2.mkdir(exist_ok=True)

drive.mount("/content/drive")
Q5_LOADER_BACKUP = (
    Path("/content/drive/MyDrive/NYU_NLP_HW4/part2/q5_checks")
    / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
)
Q5_LOADER_BACKUP.mkdir(parents=True, exist_ok=False)
print("Q5 CHECK BACKUP:", Q5_LOADER_BACKUP, flush=True)
shutil.copy2(PART2 / "load_data.py", Q5_LOADER_BACKUP / "load_data.py")
shutil.copy2(PART2 / "requirements.txt", Q5_LOADER_BACKUP / "requirements.txt")
for name in ["environment.json", "pip_freeze.txt"]:
    if (ARTIFACTS2 / name).is_file():
        shutil.copy2(ARTIFACTS2 / name, Q5_LOADER_BACKUP / name)

Q5_LOADER_SOURCE = "\"\"\"Read-only Colab check of the student's uploaded part2/load_data.py.\"\"\"\n\nimport importlib.metadata\nimport json\nfrom pathlib import Path\nimport sys\n\nimport torch\nfrom torch.utils.data import RandomSampler, SequentialSampler\n\nassert Path(\"load_data.py\").is_file(), \"Run with cwd=/content/part2.\"\nsys.path.insert(0, str(Path.cwd()))\nfrom load_data import PAD_IDX, load_t5_data, preprocess_sql, restore_sql\n\n\ntrain_loader, dev_loader, test_loader = load_t5_data(8, 8)\nloaders = {\"train\": train_loader, \"dev\": dev_loader, \"test\": test_loader}\nexpected_sizes = {\"train\": 4225, \"dev\": 466, \"test\": 432}\nassert isinstance(train_loader.sampler, RandomSampler)\nassert isinstance(dev_loader.sampler, SequentialSampler)\nassert isinstance(test_loader.sampler, SequentialSampler)\nreport = {\n    \"checkpoint\": \"google-t5/t5-small\",\n    \"batch_size_for_check\": 8,\n    \"versions\": {name: importlib.metadata.version(name)\n                 for name in [\"torch\", \"transformers\", \"tokenizers\"]},\n    \"splits\": {},\n}\n\nfor split, loader in loaders.items():\n    dataset = loader.dataset\n    assert len(dataset) == expected_sizes[split], f\"Wrong {split} example count\"\n    tokenizer = dataset.tokenizer\n    assert PAD_IDX == tokenizer.pad_token_id == 0\n    assert all(ids[-1] == tokenizer.eos_token_id for ids in dataset.input_ids)\n    batch = next(iter(loader))\n    expected_columns = 3 if split == \"test\" else 5\n    assert len(batch) == expected_columns, f\"Wrong {split} batch structure\"\n    encoder_ids, encoder_mask = batch[:2]\n    assert encoder_ids.dtype == encoder_mask.dtype == torch.long\n    assert encoder_ids.shape == encoder_mask.shape\n    assert encoder_ids.shape[0] == 8\n    assert torch.equal(encoder_mask, (encoder_ids != PAD_IDX).long())\n\n    info = {\n        \"examples\": len(dataset),\n        \"batches\": len(loader),\n        \"max_input_tokens_with_EOS\": max(map(len, dataset.input_ids)),\n        \"batch_shapes\": [list(t.shape) for t in batch],\n    }\n    if split == \"test\":\n        assert dataset.target_ids is None\n        initial = batch[2]\n        assert initial.shape == (8, 1) and torch.all(initial == PAD_IDX)\n        raw_test = Path(\"data/test.nl\").read_text().splitlines()\n        expected_ids = tokenizer(\n            [s.strip() for s in raw_test], add_special_tokens=True,\n            padding=False, truncation=False, return_attention_mask=False,\n        )[\"input_ids\"]\n        assert dataset.input_ids == expected_ids, \"Test inputs were reordered or changed.\"\n    else:\n        decoder_inputs, targets, initial = batch[2:]\n        assert decoder_inputs.shape == targets.shape\n        assert targets.dtype == decoder_inputs.dtype == initial.dtype == torch.long\n        assert torch.all(targets >= 0), \"Starter loss expects PAD=0, not -100 labels.\"\n        assert initial.shape == (8, 1)\n        assert torch.equal(initial, decoder_inputs[:, :1])\n        assert torch.all(initial == PAD_IDX)\n        assert torch.equal(decoder_inputs[:, 1:], targets[:, :-1]), (\n            \"Incorrect teacher-forcing shift.\"\n        )\n        assert all(ids[-1] == tokenizer.eos_token_id for ids in dataset.target_ids)\n        assert all(tokenizer.unk_token_id not in ids for ids in dataset.target_ids)\n        info[\"max_target_tokens_with_EOS\"] = max(map(len, dataset.target_ids))\n        expected_max = {\"train\": (60, 511), \"dev\": (44, 503)}[split]\n        assert (info[\"max_input_tokens_with_EOS\"],\n                info[\"max_target_tokens_with_EOS\"]) == expected_max, (\n            \"Loader token lengths differ from Q4 + EOS; inspect preprocessing/truncation.\"\n        )\n    report[\"splits\"][split] = info\n    print(f\"{split}: {len(dataset)} examples; {len(loader)} batches\", flush=True)\n    print(\"  batch shapes:\", info[\"batch_shapes\"], flush=True)\n\ntokenizer = train_loader.dataset.tokenizer\nfor raw in [\"x < 1200\", \"x <= 1200\", \"x > 1200\", \"x >= 1200\"]:\n    encoded = preprocess_sql(raw)\n    ids = tokenizer(encoded, add_special_tokens=True)[\"input_ids\"]\n    assert tokenizer.unk_token_id not in ids\n    decoded = tokenizer.decode(ids, skip_special_tokens=True,\n                               clean_up_tokenization_spaces=False)\n    assert restore_sql(decoded) == raw\n\nPath(sys.argv[1]).write_text(json.dumps(report, indent=2))\nprint(\"Q5 DATA LOADER CHECK PASSED\", flush=True)\nprint(\"EOS, decoder shift, padding, token coverage, and split ordering verified.\")\nprint(\"No model training has started.\")\n"

check_script = ARTIFACTS2 / "q5_check_data_loader.py"
check_script.write_text(Q5_LOADER_SOURCE)
shutil.copy2(check_script, Q5_LOADER_BACKUP / check_script.name)

def loader_check_status(state, error=None):
    status = {
        "state": state,
        "stage": "Q5_data_loader_check_only",
        "updated_utc": datetime.now(timezone.utc).isoformat(),
        "backup_folder": str(Q5_LOADER_BACKUP),
    }
    if error:
        status["error"] = error
    (Q5_LOADER_BACKUP / "status.json").write_text(json.dumps(status, indent=2))

loader_check_status("RUNNING")
try:
    with (Q5_LOADER_BACKUP / "loader_check.log").open("w", buffering=1) as log:
        run_p2_command([
            PYTHON2, "-u", str(check_script),
            str(Q5_LOADER_BACKUP / "loader_check.json"),
        ], cwd=PART2, log=log)
except BaseException:
    loader_check_status("FAILED", traceback.format_exc())
    print("Check failed. Saved log:", Q5_LOADER_BACKUP / "loader_check.log")
    raise
else:
    loader_check_status("COMPLETED")
    print("\nLOADER CHECK BACKED UP:", Q5_LOADER_BACKUP)
    print("Save this notebook and ask for review before starting model training.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Q5 CHECK BACKUP: /content/drive/MyDrive/NYU_NLP_HW4/part2/q5_checks/20261110T024358303012Z

Running: /content/hw4-part2-env/bin/python -u /content/hw4-part2-artifacts/q5_check_data_loader.py /content/drive/MyDrive/NYU_NLP_HW4/part2/q5_checks/20261110T024358303012Z/loader_check.json
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
/content/hw4-part2-env/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
train: 4225 examples; 529 batches
  batch shapes: [[8, 25], [8, 25], [8, 325], [8, 325], [8, 1]]
dev: 466 examples; 59 batches
  batch shapes: [[8, 26], [8, 26], [8, 255], [8, 255], 

## 8. Q5 next step: pretrained model and checkpoint helpers

First open `support/q5_model_utils_reference.py`. In `part2/t5_utils.py`, replace **only** `initialize_model`, `save_model`, and `load_model_from_checkpoint` with the corresponding reference functions. Keep `mkdir`, `setup_wandb`, all imports, `DEVICE`, and the optimizer/scheduler helpers. Save and upload the updated `part2` folder to Colab.

The required fine-tuning branch loads pretrained `google-t5/t5-small` and leaves all parameters trainable. Checkpoints have separate `best/` and `last/` subfolders and include the model/config and tokenizer, using safetensors. These helpers save inference-ready weights, not an optimizer/scheduler resume state.

Run only the following new code cell. It may download the T5-small weights the first time. It checks a real GPU forward/backward pass and one optimizer/scheduler step on two training examples. It tests model/tokenizer save-and-reload consistency and proves that writing `last` does not overwrite `best`.

**This is a diagnostic, not the full experiment.** Its learning rate and short scheduler are check settings, not final hyperparameters. Its temporary checkpoint files remain under a unique `/content/hw4_q5_model_check_.../` folder, outside real experiment checkpoints. Only source snapshots, logs, status, and the diagnostic JSON are saved to Drive. No development or test examples are used for the diagnostic update.

In [8]:
# Run after replacing the three model helper functions and uploading part2.
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive
import json
import shutil
import traceback

PART2 = Path("/content/part2")
PYTHON2 = "/content/hw4-part2-env/bin/python"
ARTIFACTS2 = Path("/content/hw4-part2-artifacts")
assert Path(PYTHON2).is_file(), "Use your Part 2 Colab environment."
assert callable(globals().get("run_p2_command")), "Run Cell 1 to define run_p2_command."
assert (PART2 / "t5_utils.py").is_file(), "Upload your updated part2 folder first."
ARTIFACTS2.mkdir(exist_ok=True)

drive.mount("/content/drive")
Q5_MODEL_BACKUP = (
    Path("/content/drive/MyDrive/NYU_NLP_HW4/part2/q5_model_checks")
    / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
)
Q5_MODEL_BACKUP.mkdir(parents=True, exist_ok=False)
print("MODEL CHECK LOG/REPORT FOLDER:", Q5_MODEL_BACKUP, flush=True)
for name in ["t5_utils.py", "load_data.py", "requirements.txt"]:
    shutil.copy2(PART2 / name, Q5_MODEL_BACKUP / name)
for name in ["environment.json", "pip_freeze.txt"]:
    if (ARTIFACTS2 / name).is_file():
        shutil.copy2(ARTIFACTS2 / name, Q5_MODEL_BACKUP / name)

Q5_MODEL_SOURCE = "\"\"\"Check uploaded model helpers using a temporary, non-final model experiment.\"\"\"\n\nimport gc\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nimport tempfile\nfrom types import SimpleNamespace\n\nimport torch\nfrom transformers import T5ForConditionalGeneration, T5TokenizerFast\n\npart2 = Path.cwd().resolve()\nassert (part2 / \"t5_utils.py\").is_file(), \"Run with cwd=/content/part2.\"\nsys.path.insert(0, str(part2))\nfrom load_data import T5Dataset, normal_collate_fn\nfrom t5_utils import (\n    DEVICE, initialize_model, initialize_optimizer_and_scheduler,\n    save_model, load_model_from_checkpoint,\n)\nfrom utils import set_random_seeds\n\nassert torch.cuda.is_available(), \"Connect to your Colab GPU before this model check.\"\nassert DEVICE.type == \"cuda\"\nset_random_seeds(42)\nreport_path = Path(sys.argv[1]).resolve()\n\n# These hyperparameters are for ONE diagnostic update, not the final experiment.\nargs = SimpleNamespace(\n    finetune=True, experiment_name=\"smoke_only\",\n    optimizer_type=\"AdamW\", learning_rate=3e-4, weight_decay=0.0,\n    scheduler_type=\"linear\", num_warmup_epochs=0, max_n_epochs=2,\n)\nprint(\"Loading pretrained T5-small; the first weights download may take a few minutes.\", flush=True)\nmodel = initialize_model(args)\nassert isinstance(model, T5ForConditionalGeneration), \"initialize_model returned the wrong model.\"\nassert model.config.model_type == \"t5\"\nassert model.config.d_model == 512 and model.config.num_layers == 6\nassert model.config.decoder_start_token_id == model.config.pad_token_id == 0\nassert model.config.eos_token_id == 1\nassert next(model.parameters()).device.type == \"cuda\"\ntotal = sum(p.numel() for p in model.parameters())\ntrainable = sum(p.numel() for p in model.parameters() if p.requires_grad)\nassert total == trainable\nprint(f\"Parameters: {total:,}; trainable: {trainable:,}\", flush=True)\n\ndataset = T5Dataset(str(part2 / \"data\"), \"train\")\nbatch = normal_collate_fn([dataset[0], dataset[1]])\nencoder_ids, encoder_mask, decoder_inputs, targets, _ = [t.to(DEVICE) for t in batch]\ninputs = dict(input_ids=encoder_ids, attention_mask=encoder_mask,\n              decoder_input_ids=decoder_inputs)\n\ndef probe(current_model):\n    current_model.eval()\n    with torch.no_grad():\n        # Small numerical signature: enough to check exact save/load behavior.\n        return current_model(**inputs).logits[:, :2, :64].detach().cpu().clone()\n\nbefore = probe(model)\nassert torch.isfinite(before).all()\n\n# Isolate smoke checkpoints so no real training checkpoint is overwritten.\nwork = Path(tempfile.mkdtemp(prefix=\"hw4_q5_model_check_\", dir=\"/content\"))\nos.chdir(work)\ncheckpoint_dir = Path(\"checkpoints/ft_experiments\") / args.experiment_name\nprint(\"Temporary diagnostic checkpoint directory:\", work / checkpoint_dir, flush=True)\nsave_model(str(checkpoint_dir), model, best=True)\n\nbest_dir = checkpoint_dir / \"best\"\nassert (best_dir / \"config.json\").is_file()\nassert (best_dir / \"model.safetensors\").is_file()\nsaved_tokenizer = T5TokenizerFast.from_pretrained(str(best_dir), local_files_only=True)\nassert saved_tokenizer.get_vocab() == dataset.tokenizer.get_vocab()\nrestored_best = load_model_from_checkpoint(args, best=True)\ntorch.testing.assert_close(probe(restored_best), before, rtol=1e-5, atol=1e-6)\ndel restored_best\ngc.collect()\ntorch.cuda.empty_cache()\nprint(\"BEST checkpoint and tokenizer reload: PASSED\", flush=True)\n\noptimizer, scheduler = initialize_optimizer_and_scheduler(args, model, epoch_length=1)\nassert scheduler is not None\ninitial_weights = model.shared.weight[:64, :64].detach().clone()\nmodel.train()\noptimizer.zero_grad()\nlogits = model(**inputs).logits\nnon_pad = targets != 0\nloss = torch.nn.functional.cross_entropy(logits[non_pad], targets[non_pad])\nassert torch.isfinite(loss)\nloss.backward()\nassert model.shared.weight.grad is not None\nassert torch.isfinite(model.shared.weight.grad).all()\noptimizer.step()\nscheduler.step()\noptimizer.zero_grad(set_to_none=True)\nassert not torch.equal(initial_weights, model.shared.weight[:64, :64].detach())\nafter = probe(model)\nassert not torch.equal(before, after)\nprint(f\"ONE diagnostic optimizer step: PASSED; loss={loss.item():.6f}\", flush=True)\ndel logits, initial_weights\n\nsave_model(str(checkpoint_dir), model, best=False)\nassert (checkpoint_dir / \"last\" / \"model.safetensors\").is_file()\nrestored_last = load_model_from_checkpoint(args, best=False)\ntorch.testing.assert_close(probe(restored_last), after, rtol=1e-5, atol=1e-6)\ndel restored_last\ngc.collect()\ntorch.cuda.empty_cache()\n\n# Saving last must not change the best checkpoint saved before the update.\nrestored_best = load_model_from_checkpoint(args, best=True)\ntorch.testing.assert_close(probe(restored_best), before, rtol=1e-5, atol=1e-6)\nprint(\"LAST checkpoint reload and BEST/LAST separation: PASSED\", flush=True)\n\nreport = {\n    \"stage\": \"diagnostic_model_check_not_final_training\",\n    \"pretrained_checkpoint\": \"google-t5/t5-small\",\n    \"parameters\": total, \"trainable_parameters\": trainable,\n    \"gpu\": torch.cuda.get_device_name(0),\n    \"one_step_loss\": loss.item(),\n    \"best_reload_matches\": True, \"last_reload_matches\": True,\n    \"best_and_last_are_distinct\": True, \"tokenizer_reload_matches\": True,\n    \"temporary_checkpoint_root\": str(work / checkpoint_dir),\n    \"diagnostic_args_not_final_hyperparameters\": vars(args),\n    \"versions\": {name: importlib.metadata.version(name)\n                 for name in [\"torch\", \"transformers\", \"tokenizers\"]},\n}\nreport_path.write_text(json.dumps(report, indent=2))\nprint(\"Q5 MODEL AND CHECKPOINT CHECK PASSED\", flush=True)\nprint(\"Only one diagnostic update was run. Full training has NOT started.\")\nprint(\"Diagnostic weights stay in /content; the report/log are saved to Drive.\")\n"

model_check_script = ARTIFACTS2 / "q5_check_model_utils.py"
model_check_script.write_text(Q5_MODEL_SOURCE)
shutil.copy2(model_check_script, Q5_MODEL_BACKUP / model_check_script.name)

def model_check_status(state, error=None):
    status = {
        "state": state,
        "stage": "Q5_model_checkpoint_smoke_check_only",
        "updated_utc": datetime.now(timezone.utc).isoformat(),
        "backup_folder": str(Q5_MODEL_BACKUP),
    }
    if error:
        status["error"] = error
    (Q5_MODEL_BACKUP / "status.json").write_text(json.dumps(status, indent=2))

model_check_status("RUNNING")
try:
    with (Q5_MODEL_BACKUP / "model_check.log").open("w", buffering=1) as log:
        run_p2_command([
            PYTHON2, "-u", str(model_check_script),
            str(Q5_MODEL_BACKUP / "model_check.json"),
        ], cwd=PART2, log=log)
except BaseException:
    model_check_status("FAILED", traceback.format_exc())
    print("Check failed. Saved log:", Q5_MODEL_BACKUP / "model_check.log")
    raise
else:
    model_check_status("COMPLETED")
    print("\nMODEL CHECK REPORT/LOG BACKED UP:", Q5_MODEL_BACKUP)
    print("Save this notebook and ask for review. Do not start the full training yet.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
MODEL CHECK LOG/REPORT FOLDER: /content/drive/MyDrive/NYU_NLP_HW4/part2/q5_model_checks/20261110T025729141752Z

Running: /content/hw4-part2-env/bin/python -u /content/hw4-part2-artifacts/q5_check_model_utils.py /content/drive/MyDrive/NYU_NLP_HW4/part2/q5_model_checks/20261110T025729141752Z/model_check.json
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Loading pretrained T5-small; the first weights download may take a few minutes.
/content/hw4-part2-env/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Parameters: 60,506,624; trainable: 60,506,624
Temporary diagnostic checkpoint

## 9. Q5 next step: evaluation and SQL generation

Open `support/q5_evaluation_reference.py`. In `part2/train_t5.py`, replace the existing block from `def eval_epoch(...)` through `test_inference(...)`, ending immediately BEFORE `def main():`, with the entire reference block (including its helper functions). Keep all existing imports, training functions, and `main()` unchanged for now. Save and upload your updated `part2` folder.

Run the cell below. It performs real greedy generation on **four dev examples**, checks loss, then uses controlled SQL fixtures to check marker restoration, metric ordering, SQL errors, and the NL-only inference interface. It does **not** train or use held-out test inputs. Generation may take a few minutes. Diagnostic scores are not results for your written report.

Expected ending: `Q5 EVALUATION CHECK PASSED`. Logs, diagnostic files, and a report are saved to Drive. Do not run `train_t5.py` directly yet: its starter training defaults still need configuration, and final test generation needs a separate guard.


In [9]:
# Run after pasting the evaluation reference block, saving, and uploading part2.
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive
import json
import shutil
import traceback

PART2 = Path("/content/part2")
PYTHON2 = "/content/hw4-part2-env/bin/python"
ARTIFACTS2 = Path("/content/hw4-part2-artifacts")
assert Path(PYTHON2).is_file(), "Use your Part 2 Colab environment."
assert callable(globals().get("run_p2_command")), "Run Cell 1 to define run_p2_command."
assert (PART2 / "t5_utils.py").is_file(), "Upload your updated part2 folder first."
ARTIFACTS2.mkdir(exist_ok=True)

drive.mount("/content/drive")
Q5_EVAL_BACKUP = (
    Path("/content/drive/MyDrive/NYU_NLP_HW4/part2/q5_eval_checks")
    / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
)
Q5_EVAL_BACKUP.mkdir(parents=True, exist_ok=False)
print("EVALUATION CHECK LOG/REPORT FOLDER:", Q5_EVAL_BACKUP, flush=True)
for name in ["train_t5.py", "t5_utils.py", "load_data.py", "utils.py", "requirements.txt"]:
    shutil.copy2(PART2 / name, Q5_EVAL_BACKUP / name)
for name in ["environment.json", "pip_freeze.txt"]:
    if (ARTIFACTS2 / name).is_file():
        shutil.copy2(ARTIFACTS2 / name, Q5_EVAL_BACKUP / name)

Q5_EVAL_SOURCE = "\"\"\"Check evaluation on FOUR DEV examples; never train or use held-out test inputs.\"\"\"\n\nimport json\nimport math\nfrom pathlib import Path\nimport pickle\nimport sys\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\n\nimport torch\nfrom torch.utils.data import DataLoader, Dataset, Subset\n\npart2 = Path.cwd().resolve()\nsys.path.insert(0, str(part2))\nfrom load_data import T5Dataset, normal_collate_fn, test_collate_fn, preprocess_sql\nfrom t5_utils import initialize_model\nimport train_t5 as runner\nfrom utils import save_queries_and_records, set_random_seeds\n\nassert torch.cuda.is_available(), \"Connect to a Colab GPU.\"\nassert runner.DEVICE.type == \"cuda\"\nset_random_seeds(42)\nreport_path = Path(sys.argv[1]).resolve()\nout = report_path.parent / \"diagnostic_only\"\nout.mkdir(parents=True, exist_ok=False)\nargs = SimpleNamespace(finetune=True, max_new_tokens=512, num_beams=1)\nmodel = initialize_model(args)\ndataset = T5Dataset(str(part2 / \"data\"), \"dev\")\nsubset = Subset(dataset, [0, 1, 2, 3])\nloader = DataLoader(subset, batch_size=2, shuffle=False, collate_fn=normal_collate_fn)\ntokenizer = dataset.tokenizer\n\ngold = (part2 / \"data/dev.sql\").read_text().splitlines()[:4]\ngold_path, gold_records = out / \"four_dev_gold.sql\", out / \"four_dev_gold.pkl\"\nsave_queries_and_records(gold, str(gold_path), str(gold_records))\nbefore = model.shared.weight[:16, :16].detach().clone()\n\nprint(\"Checking real greedy generation on FOUR DEV examples, not the test split.\", flush=True)\nmetrics = runner.eval_epoch(\n    args, model, loader, str(gold_path), str(out / \"four_dev_predictions.sql\"),\n    str(gold_records), str(out / \"four_dev_predictions.pkl\"),\n)\nassert len(metrics) == 5 and all(math.isfinite(float(x)) for x in metrics)\nassert metrics[0] > 0 and all(0 <= x <= 1 for x in metrics[1:])\nassert len((out / \"four_dev_predictions.sql\").read_text().splitlines()) == 4\nassert not model.training\nassert torch.equal(before, model.shared.weight[:16, :16].detach())\nprint(\"Real forward loss + greedy generation + SQL/record files: PASSED\", flush=True)\n\n# Independently recompute token-weighted, PAD-excluding loss, without generation.\ntotal_loss, total_tokens = 0.0, 0\nwith torch.no_grad():\n    for enc, mask, dec, targets, _ in loader:\n        enc, mask, dec, targets = [x.to(runner.DEVICE) for x in (enc, mask, dec, targets)]\n        logits = model(input_ids=enc, attention_mask=mask, decoder_input_ids=dec).logits\n        keep = targets != 0\n        total_loss += torch.nn.functional.cross_entropy(\n            logits[keep], targets[keep], reduction=\"sum\",\n        ).item()\n        total_tokens += keep.sum().item()\n        del logits\nassert math.isclose(metrics[0], total_loss / total_tokens, rel_tol=1e-5)\nprint(\"Token-weighted evaluation loss and PAD handling: PASSED\", flush=True)\n\n# Controlled SQL fixtures test metric order, < / <= restoration and SQL errors.\n# They are diagnostic data, not predictions or results for the assignment.\nfixture_gold = [\"SELECT 1 WHERE 1 < 2\", \"SELECT 2 WHERE 2 <= 2\", \"SELECT 3\", \"SELECT 4\"]\nfixture_predictions = fixture_gold[:3] + [\"NOT SQL\"]\nfixture_path, fixture_records = out / \"fixture_gold.sql\", out / \"fixture_gold.pkl\"\nsave_queries_and_records(fixture_gold, str(fixture_path), str(fixture_records))\n\ndef fake_generation(**kwargs):\n    assert not torch.is_grad_enabled() and not model.training\n    assert kwargs[\"decoder_input_ids\"].shape[1] == 1, \"Gold tokens leaked into generation.\"\n    assert torch.all(kwargs[\"decoder_input_ids\"] == 0)\n    assert not kwargs[\"do_sample\"] and kwargs[\"num_beams\"] == 1\n    assert kwargs[\"max_new_tokens\"] == 512\n    batch_size = kwargs[\"input_ids\"].shape[0]\n    start = fake_generation.offset\n    queries = fixture_predictions[start:start + batch_size]\n    fake_generation.offset += batch_size\n    encoded = tokenizer([preprocess_sql(q) for q in queries], padding=True, return_tensors=\"pt\")[\"input_ids\"]\n    return torch.cat([torch.zeros((batch_size, 1), dtype=torch.long), encoded], dim=1).to(runner.DEVICE)\n\nfake_generation.offset = 0\nwith patch.object(model, \"generate\", side_effect=fake_generation):\n    fixture_metrics = runner.eval_epoch(\n        args, model, loader, str(fixture_path), str(out / \"fixture_predictions.sql\"),\n        str(fixture_records), str(out / \"fixture_predictions.pkl\"),\n    )\nassert math.isclose(fixture_metrics[1], 0.75, abs_tol=1e-7), fixture_metrics\nassert fixture_metrics[2:] == (0.75, 0.75, 0.25), fixture_metrics\nassert (out / \"fixture_predictions.sql\").read_text().splitlines() == fixture_predictions\nwith (out / \"fixture_predictions.pkl\").open(\"rb\") as handle:\n    records, errors = pickle.load(handle)\nassert len(records) == len(errors) == 4 and sum(bool(e) for e in errors) == 1\nprint(\"SQL marker restoration, metric return order, error rate, file ordering: PASSED\", flush=True)\n\nclass DevInputsOnly(Dataset):\n    tokenizer = dataset.tokenizer\n    def __len__(self):\n        return len(subset)\n    def __getitem__(self, index):\n        return subset[index][0]\n\nproxy_loader = DataLoader(DevInputsOnly(), batch_size=2, shuffle=False, collate_fn=test_collate_fn)\nfake_generation.offset = 0\nwith patch.object(model, \"generate\", side_effect=fake_generation):\n    runner.test_inference(\n        args, model, proxy_loader, str(out / \"dev_inputs_only.sql\"),\n        str(out / \"dev_inputs_only.pkl\"),\n    )\nassert (out / \"dev_inputs_only.sql\").read_text().splitlines() == fixture_predictions\nwith (out / \"dev_inputs_only.pkl\").open(\"rb\") as handle:\n    records, errors = pickle.load(handle)\nassert len(records) == len(errors) == 4\nassert torch.equal(before, model.shared.weight[:16, :16].detach())\nassert all(p.grad is None for p in model.parameters())\nprint(\"NL-only inference interface on dev inputs (NOT held-out test): PASSED\", flush=True)\n\n# Refuse shuffled evaluation before any generation or output writes.\nshuffled = DataLoader(subset, batch_size=2, shuffle=True, collate_fn=normal_collate_fn)\ntry:\n    runner.eval_epoch(args, model, shuffled, str(gold_path), str(out / \"unused.sql\"),\n                      str(gold_records), str(out / \"unused.pkl\"))\nexcept ValueError:\n    pass\nelse:\n    raise AssertionError(\"Evaluation incorrectly accepted shuffled data.\")\n\nreport = {\n    \"stage\": \"evaluation_diagnostic_only_not_final_results\",\n    \"dev_examples\": 4, \"held_out_test_used\": False, \"training_updates\": 0,\n    \"real_untrained_four_dev_metrics_not_reportable\": [float(x) for x in metrics],\n    \"controlled_fixture_expected_f1_em_sql_em_error\": [0.75, 0.75, 0.75, 0.25],\n    \"checks_passed\": [\"real_generation\", \"weighted_loss\", \"sql_marker_restoration\",\n                      \"metric_order\", \"sql_error_rate\", \"output_order_and_format\",\n                      \"nl_only_interface_on_dev\", \"shuffle_guard\", \"no_training\"],\n}\nreport_path.write_text(json.dumps(report, indent=2))\nprint(\"Q5 EVALUATION CHECK PASSED\", flush=True)\nprint(\"No training updates; no held-out test inputs used. Diagnostic scores are NOT report results.\")\n"

evaluation_check_script = ARTIFACTS2 / "q5_check_evaluation.py"
evaluation_check_script.write_text(Q5_EVAL_SOURCE)
shutil.copy2(evaluation_check_script, Q5_EVAL_BACKUP / evaluation_check_script.name)

def evaluation_check_status(state, error=None):
    status = {
        "state": state,
        "stage": "Q5_evaluation_checkpoint_smoke_check_only",
        "updated_utc": datetime.now(timezone.utc).isoformat(),
        "backup_folder": str(Q5_EVAL_BACKUP),
    }
    if error:
        status["error"] = error
    (Q5_EVAL_BACKUP / "status.json").write_text(json.dumps(status, indent=2))

evaluation_check_status("RUNNING")
try:
    with (Q5_EVAL_BACKUP / "evaluation_check.log").open("w", buffering=1) as log:
        run_p2_command([
            PYTHON2, "-u", str(evaluation_check_script),
            str(Q5_EVAL_BACKUP / "evaluation_check.json"),
        ], cwd=PART2, log=log)
except BaseException:
    evaluation_check_status("FAILED", traceback.format_exc())
    print("Check failed. Saved log:", Q5_EVAL_BACKUP / "evaluation_check.log")
    raise
else:
    evaluation_check_status("COMPLETED")
    print("\nEVALUATION CHECK REPORT/LOG BACKED UP:", Q5_EVAL_BACKUP)
    print("Save this notebook and ask for review. Do not start the full training yet.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
EVALUATION CHECK LOG/REPORT FOLDER: /content/drive/MyDrive/NYU_NLP_HW4/part2/q5_eval_checks/20261110T031158388113Z

Running: /content/hw4-part2-env/bin/python -u /content/hw4-part2-artifacts/q5_check_evaluation.py /content/drive/MyDrive/NYU_NLP_HW4/part2/q5_eval_checks/20261110T031158388113Z/evaluation_check.json
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
/content/hw4-part2-env/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(

0it [00:00, ?it/s]
4it [00:00, 76.38it/s]
Checking real greedy generation on FOUR DEV examples, not the test split.

Dev loss + generation: 100%|████

In [1]:
from google.colab import drive
from pathlib import Path
import shutil
import subprocess

drive.mount("/content/drive")

print("\nGPU:")
subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    check=False
)

base = Path("/content/drive/MyDrive/NYU_NLP_HW4/part2")
previous_run = base / "training_runs/20261110T032045677993Z/source"
best_checkpoint = (
    base / "continuation_runs/20261110T140531341373Z"
    / "checkpoints/best.tar"
)

print("\nPart 2 backup folder exists:", base.is_dir())
print("Original Part 2 source backup exists:", previous_run.is_dir())
print("Previous best checkpoint is safe:", best_checkpoint.is_file())
print("Part 2 files in current runtime:", Path("/content/part2").is_dir())

free_gb = shutil.disk_usage("/content").free / (1024 ** 3)
print(f"Free Colab disk space: {free_gb:.1f} GB")

Mounted at /content/drive

GPU:

Part 2 backup folder exists: True
Original Part 2 source backup exists: True
Previous best checkpoint is safe: True
Part 2 files in current runtime: False
Free Colab disk space: 112.5 GB
